# Section V-B-1: OTEM at Different Depolarizing Noise Rates
**Fig. 14** — test pass rate vs depolarizing noise rate.
Fixed: fault infidelity ≈ 0.037 (fault_id=6), toggle probability = 1%.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # qatg package lives in ../qatg/
sys.path.insert(0, os.path.abspath('.'))    # otem.py, fault_simulators.py live here

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.circuit.library import QFT, GraphState, QuantumVolume, UnitaryGate
from qiskit.synthesis.one_qubit import OneQubitEulerDecomposer
import qiskit.circuit.library as qGate

from qatg import QATG, QATGFault
from fault_simulators import (Transient_Faulty_Backend, Permanent_Faulty_Backend,
                               get_counts_from_mem, get_em_counts_from_mem)

In [ ]:
_sx_decomposer = OneQubitEulerDecomposer('U')
class mySXFault(QATGFault):
    def __init__(self, fgate):
        super().__init__(qGate.SXGate, 0, "gateType: SX, qubits: 0")
        theta, phi, lam = _sx_decomposer.angles(fgate)
        self.fgate = qGate.UGate(theta, phi, lam)
    def createOriginalGate(self):
        return qGate.SXGate()
    def createFaultyGate(self, faultfreeGate):
        return self.fgate

In [ ]:
def get_depolarizing_backend(error_rate_1q, error_rate_2q=None):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(error_rate_1q, 1), ['sx', 'rz', 'u'])
    if error_rate_2q:
        nm.add_all_qubit_quantum_error(depolarizing_error(error_rate_2q, 2), ['cx', 'ecr', 'unitary'])
    return AerSimulator(noise_model=nm)

In [ ]:
faults     = np.load('data/faults.npy', allow_pickle=True)
ecr_faults = np.load('data/ecr_fault_list.npy', allow_pickle=True)
fault_id   = 6
sx_fault   = faults[fault_id]
print(f"Loaded {len(faults)} fault models  |  using fault_id={fault_id}")

In [ ]:
nqc        = 29
test_datas = [QuantumCircuit().from_qasm_file(f'test_circuits/test_circuit_{i}.qasm')
              for i in range(nqc)]
print(f"Loaded {len(test_datas)} test circuits")

In [ ]:
def correct_rate(counts, nqb=1):
    key = '0' * nqb
    tot = sum(counts.values())
    return counts.get(key, 0) / tot if tot > 0 else 0.0

## Select best test circuit

In [ ]:
ff_base = get_depolarizing_backend(0.001)
pf_base = Permanent_Faulty_Backend(mySXFault(sx_fault), ff_base,
                                   basis_gates=['sx','rz'], num_qubits=1)
best_tid, best_rate = 0, 0.0
for tid, qc in enumerate(test_datas):
    mem  = pf_base.run(qc, shots=10000).get_memory()
    rate = sum(1 for s in mem if '1' in s) / len(mem)
    if rate > best_rate:
        best_rate, best_tid = rate, tid
test_qc = test_datas[best_tid]
print(f"Using test circuit {best_tid}, fail rate={best_rate:.3f}")

## Sweep depolarizing noise rate (x-axis of Fig. 14)

In [ ]:
SHOTS      = 50000
SWAP_RATE  = 0.01
noise_rates = np.linspace(0.0, 0.010, 11)

res_ff, res_faulty, res_otem = [], [], []
for nr in noise_rates:
    ff_b = get_depolarizing_backend(float(nr))
    pf_b = Permanent_Faulty_Backend(mySXFault(sx_fault), ff_b,
                                    basis_gates=['sx','rz'], num_qubits=1)
    tf_b = Transient_Faulty_Backend(pf_b, ff_b, swap_rate=SWAP_RATE)

    ff_mem = ff_b.run(test_qc, shots=SHOTS, memory=True).result().get_memory()
    ft_mem = tf_b.run(test_qc, shots=SHOTS)[0]
    em_mem = tf_b.run_single_em(test_qc, test_qc, 1, 1, SHOTS)
    em_cnt = get_em_counts_from_mem(em_mem, 1, 1, 1)

    res_ff.append(correct_rate(get_counts_from_mem(ff_mem)))
    res_faulty.append(correct_rate(get_counts_from_mem(ft_mem)))
    res_otem.append(correct_rate(em_cnt))
    print(f"noise={nr:.4f}: FF={res_ff[-1]:.3f}  Faulty={res_faulty[-1]:.3f}  OTEM={res_otem[-1]:.3f}")

## Plot — Fig. 14

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(noise_rates, res_ff,    marker='o', color='tab:blue',  label='Fault-Free', s=60)
plt.scatter(noise_rates, res_faulty,marker='s', color='tab:red',   label='Faulty',     s=60)
plt.scatter(noise_rates, res_otem,  marker='^', color='tab:green', label='OTEM',       s=60)
plt.xlabel('Depolarizing Noise Rate'); plt.ylabel('Test Pass Rate')
plt.title('Fig. 14: Simulation with different depolarizing noise rates')
plt.legend(); plt.tight_layout()
plt.savefig('fig14_depolarizing_sweep.pdf', bbox_inches='tight')
plt.show()